# ASR-MDD: Mispronunciation Detection — CTC + Anti-Copy Losses

**Fixes for low F1 / copy-canonical shortcut:**
1. `WeightedRandomSampler` — oversample mispronounced samples
2. `Canonical Penalty Loss` — penalise copying canonical
3. `Contrastive Loss` — pull correct-pronunciation embeddings toward canonical, push mispronounced away

Full pipeline: Wav2Vec2 (gradual unfreeze) → SpecAugment → Audio Encoder → Fusion → CTC Head

## 1. Setup

In [2]:
import os, json, math, csv, random
import torch, torchaudio
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm import tqdm
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.nn.utils.rnn import pad_sequence
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 3. Load Wav2Vec2

In [4]:
MODEL_NAME = "nguyenvulebinh/wav2vec2-base-vietnamese-250h"
processor  = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
wav2vec2   = Wav2Vec2Model.from_pretrained(MODEL_NAME).to(device)

for p in wav2vec2.parameters():
    p.requires_grad = False
wav2vec2.eval()

AUDIO_DIM = wav2vec2.config.hidden_size   # 768
print(f"Wav2Vec2 loaded. Hidden dim: {AUDIO_DIM}")

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: nguyenvulebinh/wav2vec2-base-vietnamese-250h
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 
lm_head.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Wav2Vec2 loaded. Hidden dim: 768


## 4. Load & Merge Data

In [ ]:
DATA_DIR       = ## Link to data
AUDIO_DIR      = ## Link to audio train
AUDIO_DIR_TEST = ## Link to audio test

train_text_df  = pd.read_csv(f"{DATA_DIR}/train.csv")
train_phone_df = pd.read_csv(f"{DATA_DIR}/train_phones.csv")
test_text_df   = pd.read_csv(f"{DATA_DIR}/public_test.csv")
test_phone_df  = pd.read_csv(f"{DATA_DIR}/public_test_phones.csv")

df      = train_text_df.merge(train_phone_df, on="id", suffixes=("_text","_phoneme"))
test_df = test_text_df.merge(test_phone_df,   on="id", suffixes=("_text","_phoneme"))

def make_wav_path(base_dir, x):
    x = str(x)
    if not x.endswith(".wav"): x += ".wav"
    return os.path.join(base_dir, os.path.basename(x))

df["wav_path"]      = df["path_text"].apply(lambda x: make_wav_path(AUDIO_DIR, x))
test_df["wav_path"] = test_df["path_text"].apply(lambda x: make_wav_path(AUDIO_DIR_TEST, x))

print(f"Train: {len(df)}  |  Test: {len(test_df)}")
print(f"Missing: {sum(not os.path.exists(p) for p in df['wav_path'])} train audio files")

Train: 3180  |  Test: 718
Missing: 0 train audio files


## 5. Phoneme Vocabulary

In [6]:
all_phonemes = set()
for col in [df["canonical_phoneme"], df["transcript_phoneme"], test_df["canonical_phoneme"]]:
    for seq in col:
        all_phonemes.update(str(seq).split())

phoneme_vocab = ["<BLANK>"] + sorted(all_phonemes)
phoneme2id    = {p: i for i, p in enumerate(phoneme_vocab)}
id2phoneme    = {i: p for p, i in phoneme2id.items()}

BLANK_ID   = 0
VOCAB_SIZE = len(phoneme_vocab)
print(f"BLANK_ID={BLANK_ID} | Vocab size={VOCAB_SIZE}")

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("outputs",     exist_ok=True)
with open("checkpoints/phoneme_vocab.json", "w") as f:
    json.dump(phoneme2id, f, indent=2)

BLANK_ID=0 | Vocab size=122


In [7]:
def encode_phonemes(text):
    return [phoneme2id.get(p, BLANK_ID) for p in str(text).split()]

def decode_ctc(ids):
    """CTC greedy collapse: remove consecutive duplicates then blanks."""
    prev, out = None, []
    for i in ids:
        if i != prev:
            if i != BLANK_ID:
                out.append(id2phoneme.get(i, ""))
            prev = i
    return [p for p in out if p]

df["canonical_ids"]      = df["canonical_phoneme"].apply(encode_phonemes)
df["transcript_ids"]     = df["transcript_phoneme"].apply(encode_phonemes)
test_df["canonical_ids"] = test_df["canonical_phoneme"].apply(encode_phonemes)
test_df["transcript_ids"] = test_df["transcript_phoneme"].apply(encode_phonemes)

# Flag mispronounced samples (canonical != transcript)
df["is_mispron"] = (
    df["canonical_phoneme"].str.strip() != df["transcript_phoneme"].str.strip()
).astype(int)
test_df["is_mispron"] = (
    test_df["canonical_phoneme"].str.strip() != test_df["transcript_phoneme"].str.strip()
).astype(int)
df = pd.concat([df, test_df], ignore_index=True)
n_mispron = df["is_mispron"].sum()

print(f"Correct pronunciation : {len(df)-n_mispron} ({(len(df)-n_mispron)/len(df)*100:.1f}%)")
print(f"Mispronunciation      : {n_mispron} ({n_mispron/len(df)*100:.1f}%)")

Correct pronunciation : 3372 (86.5%)
Mispronunciation      : 526 (13.5%)


## 6. Audio Feature Extraction

In [8]:
def extract_wav2vec2_frames(wav_path: str, speed: float = 1.0) -> torch.Tensor:
    """
    Load wav → mono → 16kHz → optional speed perturbation
    → Wav2Vec2 last hidden state → [T, 768] CPU tensor.
    """
    wav, sr = torchaudio.load(wav_path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    if speed != 1.0:
        wav = torchaudio.functional.resample(wav, 16000, int(16000 * speed))

    inputs = processor(wav.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        out = wav2vec2(inputs.input_values.to(device)).last_hidden_state
    return out.squeeze(0).cpu()

## 7. SpecAugment

In [9]:
class SpecAugment(nn.Module):
    def __init__(self, time_mask=40, freq_mask=64, n_time=2, n_freq=2):
        super().__init__()
        self.time_mask = time_mask
        self.freq_mask = freq_mask
        self.n_time    = n_time
        self.n_freq    = n_freq

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [T, D]
        if not self.training: return x
        T, D = x.shape
        x = x.clone()
        for _ in range(self.n_time):
            t  = random.randint(0, min(self.time_mask, T-1))
            t0 = random.randint(0, max(T-t-1, 0))
            x[t0:t0+t, :] = 0.0
        for _ in range(self.n_freq):
            f  = random.randint(0, min(self.freq_mask, D-1))
            f0 = random.randint(0, max(D-f-1, 0))
            x[:, f0:f0+f] = 0.0
        return x

## 8. Dataset & DataLoader

In [10]:
class PhonemeDataset(Dataset):
    """Returns (audio_feat, canonical_ids, transcript_ids, is_mispron)."""
    def __init__(self, df, augment=False):
        self.df      = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        speed = random.choice([0.9, 1.0, 1.1]) if self.augment and random.random() < 0.5 else 1.0
        feat  = extract_wav2vec2_frames(row["wav_path"], speed=speed)

        canonical  = torch.tensor(row["canonical_ids"],  dtype=torch.long)
        transcript = torch.tensor(row["transcript_ids"], dtype=torch.long)
        is_mispron = torch.tensor(row["is_mispron"],     dtype=torch.float)
        return feat, canonical, transcript, is_mispron


class TestDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        feat = extract_wav2vec2_frames(row["wav_path"])
        return row["id"], feat, torch.tensor(row["canonical_ids"], dtype=torch.long)

In [11]:
def train_collate(batch):
    feats, canonicals, transcripts, misprons = zip(*batch)
    feat_lens   = torch.tensor([f.shape[0] for f in feats],       dtype=torch.long)
    target_lens = torch.tensor([t.shape[0] for t in transcripts], dtype=torch.long)
    return (
        pad_sequence(feats,       batch_first=True, padding_value=0.0),
        feat_lens,
        pad_sequence(canonicals,  batch_first=True, padding_value=BLANK_ID),
        pad_sequence(transcripts, batch_first=True, padding_value=BLANK_ID),
        target_lens,
        torch.stack(misprons),
    )

def test_collate(batch):
    ids, feats, canonicals = zip(*batch)
    feat_lens = torch.tensor([f.shape[0] for f in feats], dtype=torch.long)
    return (
        list(ids),
        pad_sequence(feats,      batch_first=True, padding_value=0.0),
        pad_sequence(canonicals, batch_first=True, padding_value=BLANK_ID),
        feat_lens,
    )

In [12]:
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["is_mispron"])
# train_df = df
# val_df = test_df
print(f"Train: {len(train_df)}  (mispron: {train_df['is_mispron'].sum()})")
print(f"Val  : {len(val_df)}    (mispron: {val_df['is_mispron'].sum()})")

BATCH_SIZE      = 8
MISPRON_WEIGHT  = 3.0   # oversample mispronounced samples N× more

# ── WeightedRandomSampler ─────────────────────────────────────────
sample_weights = train_df["is_mispron"].apply(
    lambda x: MISPRON_WEIGHT if x == 1 else 1.0
).values
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(PhonemeDataset(train_df, augment=True),
                          batch_size=BATCH_SIZE, sampler=sampler,
                          collate_fn=train_collate, num_workers=0)
val_loader   = DataLoader(PhonemeDataset(val_df, augment=False),
                          batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=train_collate, num_workers=0)
test_loader  = DataLoader(TestDataset(test_df),
                          batch_size=1, shuffle=False,
                          collate_fn=test_collate)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train: 3508  (mispron: 473)
Val  : 390    (mispron: 53)
Train batches: 439 | Val batches: 49


## 9. Model

In [13]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class CTCMDDModel(nn.Module):
    """
    Wav2Vec2 features + canonical phonemes → CTC over transcript phonemes.
    Also returns pooled hidden state for contrastive loss.
    """
    def __init__(self, vocab_size, audio_dim=768, d_model=256,
                 nhead=4, num_audio_layers=2, num_text_layers=1, dropout=0.2):
        super().__init__()

        # ── Projection + SpecAugment ───────────────────────────────────────
        self.audio_proj = nn.Sequential(nn.Linear(audio_dim, d_model), nn.LayerNorm(d_model))
        self.spec_aug   = SpecAugment()

        # ── Audio encoder ──────────────────────────────────────────────────
        self.audio_pos = PositionalEncoding(d_model, dropout=dropout)
        self.audio_enc = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead, d_model*4,
                                       dropout=dropout, batch_first=True, norm_first=True),
            num_layers=num_audio_layers)

        # ── Canonical encoder ──────────────────────────────────────────────
        self.text_emb  = nn.Embedding(vocab_size, d_model, padding_idx=BLANK_ID)
        self.text_pos  = PositionalEncoding(d_model, dropout=dropout)
        self.text_enc  = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead, d_model*4,
                                       dropout=dropout, batch_first=True, norm_first=True),
            num_layers=num_text_layers)

        # ── Fusion: audio attends to canonical ────────────────────────────
        self.fusion_attn     = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.fusion_norm     = nn.LayerNorm(d_model)
        self.fusion_ffn      = nn.Sequential(
            nn.Linear(d_model, d_model*4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model*4, d_model))
        self.fusion_ffn_norm = nn.LayerNorm(d_model)

        # ── CTC head ───────────────────────────────────────────────────────
        self.ctc_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, vocab_size))

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, audio, audio_lengths, canonical, return_hidden=False):
        B, T, _ = audio.shape
        dev = audio.device

        audio_pad_mask = torch.arange(T, device=dev)[None] >= audio_lengths[:, None]

        # ── Audio ──────────────────────────────────────────────────────────
        aug = torch.stack([self.spec_aug(audio[i]) for i in range(B)])
        a   = self.audio_pos(self.audio_proj(aug))
        a   = self.audio_enc(a, src_key_padding_mask=audio_pad_mask)

        # ── Canonical ──────────────────────────────────────────────────────
        canon_pad_mask = (canonical == BLANK_ID)
        c = self.text_pos(self.text_emb(canonical))
        c = self.text_enc(c, src_key_padding_mask=canon_pad_mask)

        # ── Fusion ─────────────────────────────────────────────────────────
        fused, _ = self.fusion_attn(a, c, c, key_padding_mask=canon_pad_mask)
        fused    = self.fusion_norm(fused + a)
        fused    = self.fusion_ffn_norm(fused + self.fusion_ffn(fused))

        log_probs = torch.log_softmax(self.ctc_head(fused), dim=-1)
        log_probs = log_probs.permute(1, 0, 2)   # [T, B, vocab]

        if return_hidden:
            # Mean-pool over valid frames → [B, d_model]
            mask   = (~audio_pad_mask).unsqueeze(-1).float()
            pooled = (fused * mask).sum(1) / mask.sum(1).clamp(min=1)
            return log_probs, pooled, c.mean(dim=1)   # also return canonical pooled

        return log_probs

## 10. Loss Functions

Three losses combined:
- **CTC loss** — main sequence prediction
- **Canonical penalty** — penalise copying canonical
- **Contrastive loss** — pull correct embeddings toward canonical, push mispron away

In [14]:
ctc_criterion = nn.CTCLoss(blank=BLANK_ID, reduction="mean", zero_infinity=True)


def canonical_penalty_loss(log_probs, canonical_ids, feat_lens, alpha=0.3):
    """
    Penalise the model for predicting the canonical sequence.
    High CTC(canonical) = model is NOT copying = good → we reward this.
    We negate and clamp to keep it stable.
    """
    canon_lens = (canonical_ids != BLANK_ID).sum(dim=1).clamp(min=1)
    ctc_canon  = ctc_criterion(log_probs, canonical_ids, feat_lens, canon_lens)
    # Negate: we WANT ctc_canon to be high (hard to predict canonical)
    return -alpha * torch.clamp(ctc_canon, max=5.0)


def contrastive_loss(audio_pooled, canon_pooled, is_mispron, margin=1.0, weight=0.2):
    """
    Correct samples  → audio embedding close to canonical embedding (dist small).
    Mispronounced    → audio embedding far from canonical embedding (dist > margin).
    """
    dist = F.pairwise_distance(audio_pooled, canon_pooled)   # [B]

    correct_mask = (is_mispron == 0).float()
    mispron_mask = is_mispron

    loss_correct = correct_mask * dist.pow(2)
    loss_mispron = mispron_mask * F.relu(margin - dist).pow(2)

    loss = (loss_correct + loss_mispron).mean()
    return weight * loss


def combined_loss(log_probs, targets, feat_lens, target_lens,
                  canonical, audio_pooled, canon_pooled, is_mispron,
                  alpha=0.3, con_weight=0.2):
    ctc  = ctc_criterion(log_probs, targets, feat_lens, target_lens)
    pen  = canonical_penalty_loss(log_probs, canonical, feat_lens, alpha=alpha)
    con  = contrastive_loss(audio_pooled, canon_pooled, is_mispron, weight=con_weight)
    return ctc + pen + con, {"ctc": ctc.item(), "penalty": pen.item(), "contrastive": con.item()}

## 11. Initialize Model

In [15]:
model = CTCMDDModel(vocab_size=VOCAB_SIZE, audio_dim=AUDIO_DIM).to(device)
print(f"Model parameters : {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable        : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

NUM_EPOCHS = 50
UNFREEZE_SCHEDULE = {10: 2, 20: 4, 30: "all"}

def get_wav2vec2_layers():
    return list(wav2vec2.encoder.layers)

def set_wav2vec2_trainable(n):
    for p in wav2vec2.parameters(): p.requires_grad = False
    layers = get_wav2vec2_layers()
    target = layers if n == "all" else layers[-n:]
    for layer in target:
        for p in layer.parameters(): p.requires_grad = True
    n_unf = len(layers) if n == "all" else n
    print(f"  Wav2Vec2: {n_unf} layer(s) unfrozen")

def make_optimizer(model_lr=3e-4, w2v_lr=1e-5):
    groups = [{"params": list(model.parameters()), "lr": model_lr}]
    w2v_params = [p for p in wav2vec2.parameters() if p.requires_grad]
    if w2v_params:
        groups.append({"params": w2v_params, "lr": w2v_lr})
    return torch.optim.AdamW(groups, weight_decay=1e-2)

optimizer = make_optimizer()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
print("Optimizer ready.")

Model parameters : 3,419,002
Trainable        : 3,419,002
Optimizer ready.


/tmp/ipykernel_24/4261667011.py:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.audio_enc = nn.TransformerEncoder(
/tmp/ipykernel_24/4261667011.py:38: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.text_enc  = nn.TransformerEncoder(


## 12. Training

In [16]:
def run_epoch(loader, is_train=True):
    model.train(is_train)
    w2v_trainable = any(p.requires_grad for p in wav2vec2.parameters())
    wav2vec2.train(is_train and w2v_trainable)

    total = {"loss": 0.0, "ctc": 0.0, "penalty": 0.0, "contrastive": 0.0}

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for audio, feat_lens, canonical, targets, target_lens, is_mispron in loader:
            audio      = audio.to(device)
            feat_lens  = feat_lens.to(device)
            canonical  = canonical.to(device)
            targets    = targets.to(device)
            target_lens= target_lens.to(device)
            is_mispron = is_mispron.to(device)

            log_probs, audio_pooled, canon_pooled = model(
                audio, feat_lens, canonical, return_hidden=True
            )

            loss, breakdown = combined_loss(
                log_probs, targets, feat_lens, target_lens,
                canonical, audio_pooled, canon_pooled, is_mispron,
                alpha=0.1, con_weight=0.05,
            )

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                all_params = list(model.parameters()) +                              [p for p in wav2vec2.parameters() if p.requires_grad]
                nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
                optimizer.step()

            total["loss"]        += loss.item()
            total["ctc"]         += breakdown["ctc"]
            total["penalty"]     += breakdown["penalty"]
            total["contrastive"] += breakdown["contrastive"]

    n = len(loader)
    return {k: v/n for k, v in total.items()}

In [17]:
best_val_loss    = float("inf")
PATIENCE         = 8
patience_counter = 0
history          = []

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Gradual unfreeze ───────────────────────────────────────────────────
    if epoch in UNFREEZE_SCHEDULE:
        n = UNFREEZE_SCHEDULE[epoch]
        print(f"\n[Epoch {epoch}] Unfreezing Wav2Vec2 ({n} layers)...")
        set_wav2vec2_trainable(n)
        optimizer = make_optimizer(model_lr=3e-4, w2v_lr=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=NUM_EPOCHS - epoch, eta_min=1e-6)

    train_m = run_epoch(train_loader, is_train=True)
    val_m   = run_epoch(val_loader,   is_train=False)
    scheduler.step()

    lr = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train {train_m['loss']:.4f} "
        f"(ctc={train_m['ctc']:.3f} pen={train_m['penalty']:.3f} con={train_m['contrastive']:.3f}) | "
        f"Val {val_m['loss']:.4f} | LR {lr:.2e}"
    )
    history.append({"epoch": epoch, **{f"train_{k}": v for k,v in train_m.items()},
                    **{f"val_{k}": v for k,v in val_m.items()}})

    if val_m["loss"] < best_val_loss:
        best_val_loss    = val_m["loss"]
        patience_counter = 0
        torch.save({
            "model_state":    model.state_dict(),
            "wav2vec2_state": wav2vec2.state_dict(),
            "phoneme2id":     phoneme2id,
            "epoch":          epoch,
            "val_loss":        val_m["loss"],
        }, "checkpoints/best_model.pt")
        print(f"  ✓ Saved (val={best_val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

pd.DataFrame(history).to_csv("checkpoints/train_history.csv", index=False)
print("\nDone. Best val loss:", best_val_loss)

Epoch 01/50 | Train 3.0864 (ctc=2.897 pen=-0.259 con=0.448) | Val 0.9359 | LR 3.00e-04
  ✓ Saved (val=0.9359)
Epoch 02/50 | Train 1.3817 (ctc=1.362 pen=-0.147 con=0.166) | Val 0.8249 | LR 2.99e-04
  ✓ Saved (val=0.8249)
Epoch 03/50 | Train 1.2247 (ctc=1.239 pen=-0.137 con=0.123) | Val 0.8273 | LR 2.97e-04
Epoch 04/50 | Train 1.1449 (ctc=1.175 pen=-0.135 con=0.104) | Val 0.8249 | LR 2.95e-04
  ✓ Saved (val=0.8249)
Epoch 05/50 | Train 1.0921 (ctc=1.133 pen=-0.132 con=0.091) | Val 0.7395 | LR 2.93e-04
  ✓ Saved (val=0.7395)
Epoch 06/50 | Train 1.0671 (ctc=1.119 pen=-0.133 con=0.082) | Val 0.7833 | LR 2.90e-04
Epoch 07/50 | Train 1.0217 (ctc=1.080 pen=-0.133 con=0.075) | Val 0.7587 | LR 2.86e-04
Epoch 08/50 | Train 0.9834 (ctc=1.045 pen=-0.132 con=0.071) | Val 0.7236 | LR 2.82e-04
  ✓ Saved (val=0.7236)
Epoch 09/50 | Train 0.9647 (ctc=1.027 pen=-0.133 con=0.070) | Val 0.7295 | LR 2.77e-04

[Epoch 10] Unfreezing Wav2Vec2 (2 layers)...
  Wav2Vec2: 2 layer(s) unfrozen
Epoch 10/50 | Train 1.43

## 13. Inference

In [ ]:
ckpt = torch.load("path/to/checkpoint.pt", map_location=device)
model.load_state_dict(ckpt["model_state"])
wav2vec2.load_state_dict(ckpt["wav2vec2_state"])
model.eval(); wav2vec2.eval()
print(f"Loaded checkpoint — epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")

Loaded checkpoint — epoch 5, val_loss=0.9497


In [20]:
@torch.no_grad()
def infer_batch(audio, feat_lens, canonical):
    log_probs = model(audio, feat_lens, canonical)   # [T, B, vocab]
    results = []
    for b in range(log_probs.size(1)):
        ids = log_probs[:, b, :].argmax(dim=-1).cpu().tolist()
        results.append(" ".join(decode_ctc(ids)))
    return results


results = []
for ids, audio, canonical, feat_lens in tqdm(test_loader, desc="Inference"):
    preds = infer_batch(
        audio.to(device), feat_lens.to(device), canonical.to(device)
    )
    for _id, pred in zip(ids, preds):
        results.append({"id": _id, "predict": pred})

df_out = pd.DataFrame(results)
df_out.to_csv("outputs/results.csv", index=False)
print(f"Saved {len(df_out)} predictions → outputs/results.csv")
df_out.head()

Inference: 100%|██████████| 718/718 [00:25<00:00, 28.56it/s]

Saved 718 predictions → outputs/results.csv


,id,predict
0,bia-dat_1618490611607,k ə-4 mz m ɛ-4 ɛ-0 mz ɨə-5 s ɔ-0 e-1 ə-4 ɛ-4 l...
1,tuoi-phia_1618720389743,t w o-3 iz f iə-1
2,tuoi-phia_1618720406706,t w o-3 iz f i-2
3,tuoi-phia_1618720417566,t w o-3 iz f iə-3
4,tuoi-phia_1618720438278,t w o-3 iz f i-2


In [21]:
pri_results = []
for ids, audio, canonical, feat_lens in tqdm(private_loader, desc="Inference"):
    preds = infer_batch(
        audio.to(device), feat_lens.to(device), canonical.to(device)
    )
    for _id, pred in zip(ids, preds):
        pri_results.append({"id": _id, "predict": pred})

pri_df_out = pd.DataFrame(pri_results)
pri_df_out.to_csv("outputs/private_results.csv", index=False)
print(f"Saved {len(df_out)} predictions → outputs/private_results.csv")

Inference: 100%|██████████| 856/856 [00:34<00:00, 25.08it/s]

Saved 718 predictions → outputs/private_results.csv


## 14. Evaluation (F1 / PER / DER)

In [22]:
def _align(seq1, seq2):
    GAP, MATCH, MISMATCH = -1, 1, -1
    n, m = len(seq1), len(seq2)
    score = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): score[i][0] = GAP*i
    for j in range(n+1): score[0][j] = GAP*j
    for i in range(1,m+1):
        for j in range(1,n+1):
            s = MATCH if seq1[j-1]==seq2[i-1] else (GAP if "<eps>" in (seq1[j-1],seq2[i-1]) else MISMATCH)
            score[i][j] = max(score[i-1][j-1]+s, score[i-1][j]+GAP, score[i][j-1]+GAP)
    a1,a2 = [],[]
    i,j = m,n
    while i>0 and j>0:
        s = MATCH if seq1[j-1]==seq2[i-1] else (GAP if "<eps>" in (seq1[j-1],seq2[i-1]) else MISMATCH)
        if   score[i][j]==score[i-1][j-1]+s: a1.append(seq1[j-1]); a2.append(seq2[i-1]); i-=1; j-=1
        elif score[i][j]==score[i][j-1]+GAP: a1.append(seq1[j-1]); a2.append("<eps>");    j-=1
        else:                                  a1.append("<eps>");    a2.append(seq2[i-1]); i-=1
    while j>0: a1.append(seq1[j-1]); a2.append("<eps>"); j-=1
    while i>0: a1.append("<eps>");    a2.append(seq2[i-1]); i-=1
    a1.reverse(); a2.reverse()
    return a1,a2

def _ops(a1,a2):
    return ["D" if r!="<eps>" and h=="<eps>" else
            "I" if r=="<eps>" and h!="<eps>" else
            "S" if r!=h else "C"
            for r,h in zip(a1,a2)]

def _align_pair(s1,s2):
    s1 = s1.replace("*","").replace("$","").split()
    s2 = s2.replace("*","").replace("$","").split()
    a1,a2 = _align(s1,s2)
    return a1,a2,_ops(a1,a2)

def _read_csv(path):
    with open(path,newline="",encoding="utf-8") as f:
        return list(csv.DictReader(f))
def compute_f1(gt_path, res_path):
    gt, res = _read_csv(gt_path), _read_csv(res_path)
    cor_cor=cor_nocor=sub_sub=sub_sub1=sub_nosub=0
    ins_ins=ins_ins1=ins_noins=del_del=del_del1=del_nodel=0
    for g,r in zip(gt,res):
        ref_seq,human_seq,op_rh = _align_pair(g["canonical"],g["transcript"])
        human_seq2,our_seq2,op_ho = _align_pair(g["transcript"],r["predict"])
        ref_seq3,our_seq3,op_ro   = _align_pair(g["canonical"],r["predict"])
        fl=0
        for i in range(len(ref_seq)):
            if ref_seq[i]=="<eps>": continue
            while fl<len(ref_seq3) and ref_seq3[fl]=="<eps>": fl+=1
            if fl<len(ref_seq3) and ref_seq[i]==ref_seq3[fl]:
                if   op_rh[i]=="D" and op_ro[fl]=="D":                del_del+=1
                elif op_rh[i]=="D" and op_ro[fl]!="D" and op_ro[fl]!="C": del_del1+=1
                elif op_rh[i]=="D" and op_ro[fl]=="C":                del_nodel+=1
                fl+=1
        fl=0
        for i in range(len(human_seq)):
            if human_seq[i]=="<eps>": continue
            while fl<len(human_seq2) and human_seq2[fl]=="<eps>": fl+=1
            if fl<len(human_seq2) and human_seq[i]==human_seq2[fl]:
                if   op_rh[i]=="C" and op_ho[fl]=="C":  cor_cor+=1
                elif op_rh[i]=="C" and op_ho[fl]!="C":  cor_nocor+=1
                if   op_rh[i]=="S" and op_ho[fl]=="C":  sub_sub+=1
                elif op_rh[i]=="S" and op_ho[fl]!="C" and ref_seq[i]!=our_seq2[fl]: sub_sub1+=1
                elif op_rh[i]=="S" and op_ho[fl]!="C" and ref_seq[i]==our_seq2[fl]: sub_nosub+=1
                if   op_rh[i]=="I" and op_ho[fl]=="C":  ins_ins+=1
                elif op_rh[i]=="I" and op_ho[fl]!="C" and op_ho[fl]!="D": ins_ins1+=1
                elif op_rh[i]=="I" and op_ho[fl]=="D":  ins_noins+=1
                fl+=1
    TR=sub_sub+sub_sub1+del_del+del_del1+ins_ins+ins_ins1
    FR=cor_nocor; FA=sub_nosub+ins_noins+del_nodel
    p=TR/(TR+FR) if TR+FR else 0.0; r=TR/(TR+FA) if TR+FA else 0.0
    return 2*p*r/(p+r) if p+r else 0.0

def compute_per(gt_path, res_path):
    gt,res=_read_csv(gt_path),_read_csv(res_path)
    S=D=I=REF=0
    for g,r in zip(gt,res):
        _,_,op=_align_pair(g["transcript"],r["predict"])
        S+=op.count("S"); D+=op.count("D"); I+=op.count("I")
        REF+=op.count("S")+op.count("D")+op.count("C")
    return (S+D+I)/REF if REF else 0.0

def compute_der(gt_path, res_path):
    gt,res=_read_csv(gt_path),_read_csv(res_path)
    sub_sub=sub_sub1=sub_nosub=ins_ins=ins_ins1=ins_noins=del_del=del_del1=del_nodel=0
    for g,r in zip(gt,res):
        ref_seq,human_seq,op_rh=_align_pair(g["canonical"],g["transcript"])
        human_seq2,our_seq2,op_ho=_align_pair(g["transcript"],r["predict"])
        ref_seq3,our_seq3,op_ro=_align_pair(g["canonical"],r["predict"])
        fl=0
        for i in range(len(ref_seq)):
            if ref_seq[i]=="<eps>": continue
            while fl<len(ref_seq3) and ref_seq3[fl]=="<eps>": fl+=1
            if fl<len(ref_seq3) and ref_seq[i]==ref_seq3[fl]:
                if op_rh[i]=="D" and op_ro[fl]=="D": del_del+=1
                elif op_rh[i]=="D" and op_ro[fl]!="D" and op_ro[fl]!="C": del_del1+=1
                elif op_rh[i]=="D" and op_ro[fl]=="C": del_nodel+=1
                fl+=1
        fl=0
        for i in range(len(human_seq)):
            if human_seq[i]=="<eps>": continue
            while fl<len(human_seq2) and human_seq2[fl]=="<eps>": fl+=1
            if fl<len(human_seq2) and human_seq[i]==human_seq2[fl]:
                if op_rh[i]=="S" and op_ho[fl]=="C": sub_sub+=1
                elif op_rh[i]=="S" and op_ho[fl]!="C" and ref_seq[i]!=our_seq2[fl]: sub_sub1+=1
                elif op_rh[i]=="S" and op_ho[fl]!="C" and ref_seq[i]==our_seq2[fl]: sub_nosub+=1
                if op_rh[i]=="I" and op_ho[fl]=="C": ins_ins+=1
                elif op_rh[i]=="I" and op_ho[fl]!="C" and op_ho[fl]!="D": ins_ins1+=1
                elif op_rh[i]=="I" and op_ho[fl]=="D": ins_noins+=1
                fl+=1
    TR=sub_sub+sub_sub1+del_del+del_del1+ins_ins+ins_ins1
    FA=sub_nosub+ins_noins+del_nodel
    DE=sub_sub1+del_del1+ins_ins1
    return DE/(TR+FA) if TR+FA else 0.0

In [ ]:
results_path      = "/kaggle/working/outputs/results.csv"
ground_truth_path = f"{DATA_DIR}/public_test_phones.csv"

f1  = compute_f1(ground_truth_path, results_path)
per = compute_per(ground_truth_path, results_path)
der = compute_der(ground_truth_path, results_path)
score = 0.5*f1 + 0.4*(1-der) + 0.1*(1-per)

print(f"F1    : {f1:.4f}   contribution → {0.5*f1:.4f}")
print(f"DER   : {der:.4f}  contribution → {0.4*(1-der):.4f}")
print(f"PER   : {per:.4f}  contribution → {0.1*(1-per):.4f}")
print(f"─────────────────────────────────")
print(f"Score : {score:.4f}")